# `Chains in LangChain`

### Introduction
- Chains can help us in building pipelines in our AI Application such that everything will be chained together
- eg. My task is that LLM - OpenAI
- English - Open Ai (Translation) - Hindi - Gemini (Summary) --> Output
  

### Parallel Chains
- input - LLM1 and LLM2 --> Generated R1 and R2 --> Combined them --> Output
- Conditional Chains provides output based on a condition being true

# `Detailed Notes `

# Chains in LangChain

## 1. What is a Chain?

A **chain** in LangChain is a sequence of operations where the **output of one step becomes the input to the next step**.

In simple words:

> **Chain = Connect multiple components together to create a workflow.**

For example:

```text
User Input
    ↓
Prompt
    ↓
LLM
    ↓
Output Parser
    ↓
Final Output
```

Instead of manually calling each component, we can compose them into a single workflow.

LangChain's current architecture uses **Runnable interfaces and composition** to build these pipelines. The `|` operator is commonly used to create sequential pipelines.

---

# 2. Why Do We Need Chains?

Suppose we want to build a simple application that translates English into Hindi.

Without a chain:

```text
1. Create prompt
2. Format prompt
3. Call LLM
4. Receive response
5. Parse response
6. Return result
```

We have to manage every step manually.

With a chain:

```text
Input
  ↓
Prompt
  ↓
LLM
  ↓
Output
```

We compose these steps together.

### Main Benefit

> **Chains make multi-step LLM workflows reusable, composable, and easier to maintain.**

---

# 3. Basic Chain Structure

A simple LangChain pipeline looks like:

```text
Input
  ↓
Prompt Template
  ↓
Model
  ↓
Output Parser
  ↓
Final Output
```

For example:

```text
Topic
 ↓
Prompt
 ↓
Chat Model
 ↓
String Parser
 ↓
Explanation
```

---

# 4. Simple Example

Suppose we want an AI that explains any topic.

### Step 1 — Prompt

```text
Explain {topic} in simple language.
```

### Step 2 — Model

The model generates the explanation.

### Step 3 — Parser

Convert the model response into the required output format.

So:

```text
topic
  ↓
Prompt
  ↓
LLM
  ↓
Parser
  ↓
Answer
```

---

# 5. Creating a Chain in Modern LangChain

A common pattern is **Runnable composition** using the pipe `|` operator.

```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.chat_models import init_chat_model

model = init_chat_model("openai:gpt-5.5")

prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in simple language."
)

parser = StrOutputParser()

chain = prompt | model | parser
```

Now we can invoke the complete pipeline:

```python
response = chain.invoke({
    "topic": "RAG"
})

print(response)
```

The flow is:

```text
{"topic": "RAG"}
        ↓
ChatPromptTemplate
        ↓
Chat Model
        ↓
StrOutputParser
        ↓
Final String
```

This composition style is based on LangChain's **Runnable** abstraction.

---

# 6. What is Runnable?

## Definition

A **Runnable** is a component that can be invoked as part of a LangChain workflow.

Examples include:

* Prompt templates
* Chat models
* Output parsers
* Runnable functions
* Runnable sequences
* Runnable maps

The important idea is that these components can be **composed together**.

```text
Runnable
   ↓
Runnable
   ↓
Runnable
```

This gives LangChain a common interface for building pipelines.

---

# 7. The `|` Operator

One of the most important things to understand for interviews is:

```python
chain = prompt | model | parser
```

The `|` operator means:

> **Pass the output of the component on the left into the component on the right.**

So:

```python
prompt | model | parser
```

means:

```text
Prompt
  ↓
Model
  ↓
Parser
```

This is called **Runnable composition**.

---

# 8. Chain Example: Translation

Let's build a translation pipeline.

```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    "Translate the following English text into Hindi:\n\n{text}"
)

chain = prompt | model | StrOutputParser()

response = chain.invoke({
    "text": "Artificial Intelligence is changing the world."
})

print(response)
```

Flow:

```text
English Text
     ↓
Prompt Template
     ↓
Chat Model
     ↓
Output Parser
     ↓
Hindi Translation
```

---

# 9. Chain Example: Summarization

```python
prompt = ChatPromptTemplate.from_template(
    """
    Summarize the following text in 5 bullet points:

    {text}
    """
)

chain = prompt | model | StrOutputParser()

result = chain.invoke({
    "text": article_text
})
```

Flow:

```text
Article
  ↓
Summarization Prompt
  ↓
LLM
  ↓
Parser
  ↓
5 Bullet Points
```

---

# 10. Sequential Chain

A **sequential chain** means one operation happens after another.

Example:

```text
Input
 ↓
Generate Content
 ↓
Summarize Content
 ↓
Translate Summary
 ↓
Final Output
```

For example:

```text
User Topic
    ↓
Chain 1: Generate Explanation
    ↓
Chain 2: Summarize Explanation
    ↓
Chain 3: Translate Summary
    ↓
Final Answer
```

Conceptually:

```text
A → B → C → D
```

The output of A becomes the input to B.

---

# 11. Multi-Step Chain Example

Suppose the user asks:

> "Explain Machine Learning and translate the explanation into Hindi."

We can design:

```text
User Topic
    ↓
Generate Explanation
    ↓
Summarize
    ↓
Translate
    ↓
Final Output
```

### Chain

```python
explanation_prompt = ChatPromptTemplate.from_template(
    "Explain {topic} for a beginner."
)

summary_prompt = ChatPromptTemplate.from_template(
    "Summarize this explanation in 5 bullet points:\n\n{text}"
)

translation_prompt = ChatPromptTemplate.from_template(
    "Translate this into Hindi:\n\n{text}"
)

chain = (
    explanation_prompt
    | model
    | StrOutputParser()
    | summary_prompt
    | model
    | StrOutputParser()
    | translation_prompt
    | model
    | StrOutputParser()
)
```

Conceptually:

```text
Topic
 ↓
Prompt 1
 ↓
LLM
 ↓
Explanation
 ↓
Prompt 2
 ↓
LLM
 ↓
Summary
 ↓
Prompt 3
 ↓
LLM
 ↓
Hindi Translation
```

---

# 12. Chain vs LLM

These are different.

## LLM

The LLM is the **model that generates the response**.

```text
Input → LLM → Output
```

## Chain

A chain can contain **multiple components**, including an LLM.

```text
Input
 ↓
Prompt
 ↓
LLM
 ↓
Parser
 ↓
Output
```

Therefore:

> **LLM is one component; a chain is a workflow composed of components.**

---

# 13. Chain vs Prompt

### Prompt

Provides instructions.

```text
"Explain {topic}."
```

### Chain

Connects the prompt with other components.

```text
Prompt → Model → Parser
```

So:

> **Prompt tells the model what to do. Chain defines how multiple steps work together.**

---

# 14. Chain vs Agent

This is a very important interview question.

## Chain

The workflow is generally **predefined**.

```text
A → B → C → D
```

Example:

```text
Question
 ↓
Prompt
 ↓
LLM
 ↓
Parser
 ↓
Answer
```

The developer defines the sequence.

---

## Agent

An agent can dynamically decide what action/tool to take.

```text
             ┌→ Search
             │
User → Agent ├→ Database
             │
             └→ Calculator
```

The model determines which tool or action is appropriate.

### Easy Difference

> **Chain = Developer defines the workflow.**

> **Agent = Model can decide the next action within the workflow.**

---

# 15. Chain vs LangGraph

Another important interview distinction.

### Chain

Best for relatively straightforward workflows:

```text
A → B → C → D
```

### LangGraph

Better for complex, stateful workflows:

```text
          ┌→ Tool A
          ↓
Input → Agent → Tool B
          ↑       ↓
          └── Decision
```

LangGraph is designed for more complex, stateful agent orchestration.

### Simple Memory Trick

```text
Chain
→ Fixed sequence

LangGraph
→ Graph-based workflow

Agent
→ Dynamic decision making
```

---

# 16. Chain in RAG

Chains are very useful for RAG applications.

A basic RAG workflow can be:

```text
User Question
      ↓
Retriever
      ↓
Relevant Documents
      ↓
Prompt
      ↓
LLM
      ↓
Answer
```

This can be composed as a pipeline.

Conceptually:

```text
Question
   ↓
Retriever
   ↓
Context
   ↓
Prompt
   ↓
Chat Model
   ↓
Parser
   ↓
Answer
```

This is one of the most important practical applications of chains.

---

# 17. Chain with Retriever

For example:

```python
rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | model
    | StrOutputParser()
)
```

Conceptually:

```text
                    ┌→ Retriever → Context
                    │
Question ────────────┤
                    ↓
                  Prompt
                    ↓
                  Model
                    ↓
                  Parser
                    ↓
                 Answer
```

Notice something interesting here:

The question is used in **two places**:

```text
Question
 ├──→ Retriever
 │
 └──→ Prompt
```

This is where Runnable composition becomes particularly powerful.

---

# 18. Parallel Execution

Not every chain has to be purely sequential.

Sometimes we want:

```text
             ┌→ Summarize
Input ───────┤
             └→ Translate
```

Both operations can potentially execute independently.

Conceptually:

```text
             ┌→ Chain A
Input ───────┤
             └→ Chain B
```

LangChain provides runnable composition primitives for sequential and parallel workflows.

---

# 19. Chain Composition

One of the biggest advantages of LangChain is **composability**.

We can create small components:

```text
Prompt A
Model
Parser
```

Then combine them:

```text
Prompt A → Model → Parser
```

Then combine that pipeline with another pipeline:

```text
Chain A
   ↓
Chain B
   ↓
Chain C
```

This makes complex applications easier to build from smaller reusable components.

---

# 20. Error Handling and Reliability

In production applications, a chain may need:

* Retries
* Fallback models
* Validation
* Timeouts
* Error handling
* Logging
* Monitoring

For example:

```text
Primary Model
      ↓
   Failure?
      ↓
Fallback Model
```

LangChain's runnable abstractions support composition patterns that can be used to build these kinds of resilient workflows.

---

# 21. Why Chains Are Important in GenAI

Chains are important because real GenAI applications rarely consist of:

```text
User → LLM → Answer
```

Production applications often look more like:

```text
User
 ↓
Input Validation
 ↓
Prompt
 ↓
Retriever
 ↓
Context
 ↓
LLM
 ↓
Output Validation
 ↓
Final Response
```

Chains allow us to organize these steps into reusable workflows.

---

# 22. Important Interview Questions

## Beginner

### Q1. What is a chain in LangChain?

**Answer:**

A chain is a composition of multiple operations where the output of one step is passed to the next step to create an LLM application workflow.

---

### Q2. What is the basic structure of a chain?

**Answer:**

```text
Input → Prompt → Model → Parser → Output
```

---

### Q3. What does the `|` operator mean in LangChain?

**Answer:**

It is used for Runnable composition. The output of the component on the left becomes the input to the component on the right.

```python
chain = prompt | model | parser
```

---

### Q4. What is a Runnable?

**Answer:**

A Runnable is a component that follows LangChain's common execution interface and can be composed with other Runnables.

---

## Intermediate

### Q5. What is the difference between a chain and an LLM?

**Answer:**

An LLM is an AI model that generates or processes information.

A chain is a workflow that can contain an LLM along with prompts, retrievers, parsers, and other components.

---

### Q6. What is the difference between a chain and an agent?

**Answer:**

A chain follows a predefined workflow, while an agent can dynamically decide which tools or actions to use.

```text
Chain:
A → B → C

Agent:
A → decide → B/C/D
```

---

### Q7. Why are chains useful in RAG?

**Answer:**

Chains can connect the retrieval process with prompt construction, the LLM, and output parsing.

```text
Question
 ↓
Retriever
 ↓
Context
 ↓
Prompt
 ↓
LLM
 ↓
Answer
```

---

### Q8. What is chain composition?

**Answer:**

Chain composition means combining multiple Runnable components or smaller chains into a larger workflow.

---

# 23. Scenario-Based Interview Questions

### Q9. You need to translate a sentence, summarize the translation, and then generate a title. Would you use a chain or agent?

**Answer:**

A **chain** is appropriate because the workflow is predictable:

```text
Translation
 ↓
Summary
 ↓
Title
```

There is no need for dynamic tool selection.

---

### Q10. Your application needs to decide whether to search Google, query a database, or use a calculator. Chain or agent?

**Answer:**

An **agent** is more appropriate because the model needs to dynamically decide which tool to use.

---

### Q11. You need a simple RAG application. Can chains be used?

**Answer:**

Yes. A RAG pipeline can be composed using a retriever, prompt, model, and output parser.

---

### Q12. Your workflow has complex branching, loops, state, and human approval. Would you use a simple chain?

**Answer:**

A simple chain may become difficult to manage. **LangGraph** is more appropriate for complex, stateful, branching workflows.

---

# Key Takeaways

```text
Chain
↓
A sequence/composition of operations

Runnable
↓
Composable LangChain component

|
↓
Pass output of one Runnable to the next

Example:
Prompt | Model | Parser

Chain
↓
Good for predictable workflows

Agent
↓
Good for dynamic tool selection

LangGraph
↓
Good for complex, stateful workflows
```

---

# 3. 30-Second Revision

> **Chain = A predefined workflow of multiple components.**

Remember:

```text
Input
 ↓
Prompt
 ↓
Model
 ↓
Parser
 ↓
Output
```

### Most Important Syntax

```python
chain = prompt | model | parser
```

`|` means:

> **Output of the left component → input of the right component.**

### Chain vs Agent

```text
Chain → Fixed workflow
Agent → Dynamic decisions
```

### Chain vs LangGraph

```text
Chain      → Simple/predictable workflow
LangGraph  → Complex/stateful workflow
```

---

# 4. 2-Minute Revision

## What is a Chain?

A chain connects multiple LangChain components into a workflow.

```text
A → B → C → D
```

## Example

```python
chain = prompt | model | parser
```

Flow:

```text
Input
 ↓
Prompt
 ↓
Chat Model
 ↓
Output Parser
 ↓
Answer
```

## Runnable

LangChain's **Runnable** abstraction allows components to be composed into pipelines.

## Sequential Chain

```text
Generate
 ↓
Summarize
 ↓
Translate
```

## RAG Chain

```text
Question
 ↓
Retriever
 ↓
Context
 ↓
Prompt
 ↓
LLM
 ↓
Parser
 ↓
Answer
```

## Chain vs Agent

**Chain:**

```text
A → B → C
```

Developer controls the sequence.

**Agent:**

```text
A → Decide → Tool A/B/C
```

The model can dynamically choose actions.

## Chain vs LangGraph

Use a chain for relatively straightforward, predictable pipelines. Use LangGraph when you need more complex orchestration involving state, branching, loops, persistence, or human-in-the-loop behavior.

### Final Interview Answer

> **In LangChain, a chain is a composable workflow where multiple components such as prompts, models, retrievers, and output parsers are connected together. In modern LangChain, this is commonly done using the Runnable interface and the `|` operator. Chains are ideal for predictable workflows, while agents are useful when the model needs to make dynamic decisions, and LangGraph is better suited for complex stateful orchestration.** ([docs.langchain.com][1])

[1]: https://docs.langchain.com/oss/python/deepagents/models?utm_source=chatgpt.com "Models - Docs by LangChain"
